# Tugas Pendahuluan — Graf Sains Data
Nama: Dimas Reza Perdana · NIM: 23031554130

## 1.1 Persiapan dan Perkenalan Lingkungan
Virtual environment dipakai untuk mengisolasi paket proyek ini dari proyek lain di laptop, sehingga versi paket tidak saling mengganggu.

| Langkah di modul | Yang dijalankan (Windows PowerShell) | Keterangan |
|---|---|---|
| `python -m venv myenv` | `python -m venv myvenv` | membuat virtual environment |
| `conda deactivate` | – | tidak diperlukan karena tidak memakai conda |
| `source myvenv/bin/activate` | `myvenv\Scripts\Activate.ps1` | perintah aktivasi di Windows berbeda dari Linux/macOS |
| `uv pip install -r requirements.txt` | `pip install neo4j pandas numpy scipy networkx matplotlib tqdm ipykernel` | lihat catatan |

**Catatan versi driver.** Requirements pada modul memasang `neo4j==4.2.1`, padahal `driver.execute_query` dan
`RoutingControl` yang dipakai pada tutorial 1.2 baru tersedia mulai driver versi 5.5 (versi 4.2.1 gagal dengan
`ImportError: cannot import name 'RoutingControl'`). Karena itu digunakan **neo4j driver 6.4.1**. Paket `imdbpy`,
`cinemagoerng`, `imdbinfo`, dan `nose` tidak dipakai pada tugas ini sehingga tidak dipasang.

In [7]:
import sys, neo4j, pandas
print("Python :", sys.version.split()[0])
print("Lokasi :", sys.executable)
print("neo4j  :", neo4j.__version__)
print("pandas :", pandas.__version__)

Python : 3.11.9
Lokasi : c:\uts graf\myvenv\Scripts\python.exe
neo4j  : 6.4.1
pandas : 3.0.6


## 1.2 Mempelajari Penggunaan Driver Neo4j
Tutorial dijalankan pada database terpisah `tutorial` di instance yang sama, agar simpul `Person`
tidak tercampur dengan data RetailRocket di database `neo4j`.

In [8]:
from neo4j import GraphDatabase, RoutingControl

URI = "neo4j://127.0.0.1:7687"
AUTH = ("neo4j", open("password.txt").read().strip())
DB_TUTORIAL = "tutorial"

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.execute_query(f"CREATE DATABASE {DB_TUTORIAL} IF NOT EXISTS WAIT", database_="system")
    records, _, _ = driver.execute_query("SHOW DATABASES YIELD name, currentStatus", database_="system")
    for r in records:
        print(r["name"], "-", r["currentStatus"])

neo4j - online
system - online
tutorial - online


In [9]:
def add_friend(driver, name, friend_name):
    driver.execute_query(
        "MERGE (a:Person {name: $name}) "
        "MERGE (friend:Person {name: $friend_name}) "
        "MERGE (a)-[:KNOWS]->(friend)",
        name=name, friend_name=friend_name, database_=DB_TUTORIAL,
    )

def print_friends(driver, name):
    records, _, _ = driver.execute_query(
        "MATCH (a:Person)-[:KNOWS]->(friend) WHERE a.name = $name "
        "RETURN friend.name ORDER BY friend.name",
        name=name, database_=DB_TUTORIAL, routing_=RoutingControl.READ,
    )
    for record in records:
        print(record["friend.name"])

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    add_friend(driver, "Arthur", "Guinevere")
    add_friend(driver, "Arthur", "Lancelot")
    add_friend(driver, "Arthur", "Merlin")
    print_friends(driver, "Arthur")

Guinevere
Lancelot
Merlin


In [10]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    # Q1: database yang dipakai query
    records, summary, _ = driver.execute_query("MATCH (p:Person) RETURN count(p) AS n", database_=DB_TUTORIAL)
    print("Q1. Query dijalankan pada database:", summary.database, "| jumlah Person:", records[0]["n"])

    # Q2: mengakses database lain dari koneksi yang sama dengan klausa USE
    records, _, _ = driver.execute_query("USE neo4j MATCH (i:Item) RETURN count(i) AS n", database_=DB_TUTORIAL)
    print("Q2. USE neo4j -> jumlah Item di database neo4j:", records[0]["n"])

    # Q3: arah relasi dan derajat setiap simpul
    records, _, _ = driver.execute_query("""
        MATCH (p:Person)
        RETURN p.name AS nama,
               COUNT { (p)-[:KNOWS]->() } AS out_degree,
               COUNT { (p)<-[:KNOWS]-() } AS in_degree
        ORDER BY nama""", database_=DB_TUTORIAL)
    print("Q3. Derajat simpul:")
    for r in records:
        print(f"    {r['nama']:<10} out = {r['out_degree']}  in = {r['in_degree']}")

    r1, _, _ = driver.execute_query("MATCH (:Person {name:'Merlin'})-[:KNOWS]->(f) RETURN f.name AS n", database_=DB_TUTORIAL)
    r2, _, _ = driver.execute_query("MATCH (:Person {name:'Merlin'})-[:KNOWS]-(f) RETURN f.name AS n", database_=DB_TUTORIAL)
    print("    Teman Merlin, pola berarah (->)  :", [r["n"] for r in r1])
    print("    Teman Merlin, pola tak berarah (-):", [r["n"] for r in r2])

Q1. Query dijalankan pada database: tutorial | jumlah Person: 4
Q2. USE neo4j -> jumlah Item di database neo4j: 184033
Q3. Derajat simpul:
    Arthur     out = 3  in = 0
    Guinevere  out = 0  in = 1
    Lancelot   out = 0  in = 1
    Merlin     out = 0  in = 1
    Teman Merlin, pola berarah (->)  : []
    Teman Merlin, pola tak berarah (-): ['Arthur']


**1. Apa nama database yang digunakan pada tutorial ini?**
Tutorial pada modul memakai database **`neo4j`**, yaitu database bawaan (default) Neo4j, yang ditentukan lewat parameter
`database_="neo4j"` pada `execute_query`. Pada notebook ini tutorial dipindahkan ke database `tutorial` agar tidak
tercampur dengan data RetailRocket; `summary.database` membuktikan query benar-benar berjalan di database tersebut.

**2. Akses lain (DDL/DML Cypher) untuk mengakses database lain dalam 1 instance:**
- Klausa **`USE`**: `USE namadb MATCH (n) RETURN n`, seperti ditunjukkan pada sel di atas (`USE neo4j`).
- Perintah **`:use namadb`** di Neo4j Browser / cypher-shell untuk berpindah database aktif.
- Perintah administrasi pada database `system`: `SHOW DATABASES`, `CREATE DATABASE`, `START DATABASE`,
  `STOP DATABASE`, `DROP DATABASE` (dipakai untuk membuat database `tutorial`).
- Dari driver Python: parameter `database_="namadb"` pada `execute_query` atau `driver.session(database="namadb")`.
- *Composite database* (Enterprise): menggabungkan beberapa database dalam satu query, misalnya `USE gabungan.db1`.

Catatan: Neo4j Community Edition hanya mendukung satu database pengguna (selain `system`), sehingga `CREATE DATABASE`
membutuhkan Enterprise Edition.

**3. Apa jenis graf pertemanan yang telah dibuat?**
- **Graf berarah (directed):** relasi `(a)-[:KNOWS]->(friend)` hanya dari Arthur ke temannya. Arthur memiliki
  out-degree 3, sedangkan Guinevere, Lancelot, dan Merlin masing-masing in-degree 1 dan out-degree 0.
  Akibatnya, dengan pola berarah Merlin tidak memiliki teman, padahal pertemanan seharusnya timbal balik.
- **Tak berbobot:** relasi tidak memiliki properti bobot.
- **Graf sederhana:** `MERGE` mencegah simpul dan relasi ganda.
- **Labeled property graph:** simpul berlabel `Person` dengan properti `name`.
- Secara struktur berbentuk **graf bintang (star graph)** dengan Arthur sebagai pusat (4 simpul, 3 sisi). Jika arah
  diabaikan, graf ini adalah **K₁,₃**, yaitu sebuah pohon (tree).

### CYPHER Total Reset
Perintah truncate dijalankan hanya pada database `tutorial`. Jika dijalankan pada database `neo4j`, seluruh data
RetailRocket akan ikut terhapus; inilah alasan modul memperingatkan penggunaannya pada *shared database*.

In [11]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    driver.execute_query("MATCH (n) DETACH DELETE n", database_=DB_TUTORIAL)
    records, _, _ = driver.execute_query("MATCH (n) RETURN count(n) AS n", database_=DB_TUTORIAL)
    print("Simpul tersisa di database tutorial:", records[0]["n"])

Simpul tersisa di database tutorial: 0


## 1.3 Melakukan Proses Impor Data dari RetailRocket ke Database Neo4J
Sumber data: Roman Zykov (Kaggle, `retailrocket/ecommerce-dataset`). Data diunduh di Windows PowerShell dengan:

`curl.exe -L -o ecommerce-dataset.zip https://www.kaggle.com/api/v1/datasets/download/retailrocket/ecommerce-dataset`

lalu `events.csv` diekstrak ke folder `data/`. Fitur pada `events.csv`:

| Kolom | Keterangan |
|---|---|
| `timestamp` | waktu kejadian dalam format Unix epoch **milidetik** (UTC) |
| `visitorid` | ID unik pengunjung |
| `event` | jenis interaksi: `view`, `addtocart`, `transaction` |
| `itemid` | ID unik produk |
| `transactionid` | ID transaksi, hanya terisi jika event berjenis `transaction` |

In [ ]:
import pandas as pd

df = pd.read_csv("events.csv")
print("Jumlah baris:", len(df))
print(df.dtypes)
df.groupby("event").head(1)      # satu contoh baris untuk setiap jenis event

Jumlah baris: 2756101
timestamp          int64
visitorid          int64
event                str
itemid             int64
transactionid    float64
dtype: object


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
17,1433223236124,287857,addtocart,5206,NaN
130,1433222276276,599528,transaction,356475,4000.0
